# CNN78u: fixed 20% sampling and optimizer diagnosis

This notebook ends before residual networks or other architecture comparisons.
It preserves the original four-block CNN and compares Adam weight decay 0.0001
against 0, using exactly the same fixed training subset, initialization and
per-epoch ordering. Learning rate 0.001, dropout 0.5, batch size 64 and three epochs
are identical. This is an optimizer diagnostic, not a claim that removing weight
decay improves generalization.

Upload **this notebook only** beside the existing `cnn78u_colab_bundle.zip` in
`MyDrive/Colab Notebooks/EC/Project/Demo`. The helper source is embedded below;
no replacement of the original notebook or bundle is needed. Select a GPU runtime.

Only training and validation data are evaluated. There is no test-evaluation cell.
The already inspected historical test period is not a fresh holdout.


## 1. Dependencies

Keep Colab's preinstalled CUDA-enabled PyTorch. Install only missing reporting/data packages. Python code and outputs are in English. The local-test branch is for repository QA and does not mount Drive or evaluate the real held-out test set.

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path
LOCAL_TEST = os.environ.get("CNN78U_LOCAL_NOTEBOOK_TEST") == "1"
packages = {"numpy":"numpy", "pandas":"pandas", "pyarrow":"pyarrow", "PIL":"Pillow",
            "sklearn":"scikit-learn", "matplotlib":"matplotlib", "tqdm":"tqdm", "psutil":"psutil"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch is missing. Select a standard Colab GPU runtime.")

## 2. Mount Drive

Authentication is performed by Colab. No username, password, token, or service-account key is embedded in this notebook.

In [ ]:
if LOCAL_TEST:
    DRIVE_DEMO = Path(os.environ["CNN78U_LOCAL_DEMO"])
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DEMO = Path("/content/drive/MyDrive/Colab Notebooks/EC/Project/Demo")
assert DRIVE_DEMO.is_dir(), f"Upload files first: {DRIVE_DEMO}"
print("Drive project:", DRIVE_DEMO)

## 3. Stage data on the runtime disk and verify the upload

Training reads numeric memmaps from the VM's local disk, not thousands of small Drive reads. The preferred upload is the provided ZIP. An expanded bundle or project tree with `us_pipeline/` and the current prepared directory is also accepted. Copy/unpack/hash time is measured separately from training.

[Colab's official FAQ](https://research.google.com/colaboratory/faq.html) recommends avoiding many small Drive I/O operations. Runtime storage is temporary; checkpoints and reports are copied back to Drive.

In [ ]:
import hashlib, json, shutil, tempfile, time, zipfile
staging_start = time.perf_counter()
cache_parent = None if LOCAL_TEST else "/content"
LOCAL_CACHE = Path(tempfile.mkdtemp(prefix="cnn78u_", dir=cache_parent))
LOCAL_PROJECT = LOCAL_CACHE / "project"
LOCAL_PROJECT.mkdir()
archive_candidates = [DRIVE_DEMO / "cnn78u_colab_bundle.zip",
                      DRIVE_DEMO / "colab/cnn78u_colab_bundle.zip",
                      DRIVE_DEMO / "ImageTrading-CV-Project/colab/cnn78u_colab_bundle.zip"]
archive_source = next((p for p in archive_candidates if p.is_file()), None)
if archive_source:
    archive = LOCAL_CACHE / "bundle.zip"
    shutil.copy2(archive_source, archive)
    with zipfile.ZipFile(archive) as z:
        required = sum(info.file_size for info in z.infolist())
        if required > shutil.disk_usage(LOCAL_CACHE).free:
            raise RuntimeError("Not enough runtime disk space to unpack the bundle")
        for info in z.infolist():
            destination = (LOCAL_PROJECT / info.filename).resolve()
            if not destination.is_relative_to(LOCAL_PROJECT.resolve()):
                raise ValueError("Unsafe archive member path")
        z.extractall(LOCAL_PROJECT)
else:
    candidates = [DRIVE_DEMO, DRIVE_DEMO / "cnn78u_colab_bundle", DRIVE_DEMO / "ImageTrading-CV-Project"]
    source = next((p for p in candidates if (p / "us_pipeline/models.py").is_file()), None)
    if source is None:
        raise FileNotFoundError("Upload cnn78u_colab_bundle.zip next to this notebook")
    prepared_source = next((p for p in [source / "prepared", source / "artifacts/cnn78u_5min_h1/prepared"]
                            if (p / "manifest.json").is_file()), None)
    if prepared_source is None:
        raise FileNotFoundError("Missing prepared next-bar dataset; raw bars alone are insufficient here")
    shutil.copytree(source / "us_pipeline", LOCAL_PROJECT / "us_pipeline", ignore=shutil.ignore_patterns("__pycache__"))
    shutil.copytree(prepared_source, LOCAL_PROJECT / "prepared")
    if (source / "bundle_manifest.json").exists():
        shutil.copy2(source / "bundle_manifest.json", LOCAL_PROJECT / "bundle_manifest.json")
verified_upload = False
if (LOCAL_PROJECT / "bundle_manifest.json").exists():
    upload = json.loads((LOCAL_PROJECT / "bundle_manifest.json").read_text())
    for relative, details in upload["files"].items():
        path = (LOCAL_PROJECT / relative).resolve()
        assert path.is_relative_to(LOCAL_PROJECT.resolve()), relative
        assert hashlib.sha256(path.read_bytes()).hexdigest() == details["sha256"], relative
    verified_upload = True
sys.path.insert(0, str(LOCAL_PROJECT))
PREPARED = LOCAL_PROJECT / "prepared"
STAGING_SECONDS = time.perf_counter() - staging_start
print({"local_project": str(LOCAL_PROJECT), "staging_seconds": round(STAGING_SECONDS, 2),
       "upload_checksums_verified": verified_upload})

## 4. Load the diagnostic runner
The helper is written to the temporary runtime directory. The uploaded bundle and
original run are preserved. GPU runs use full precision and deterministic cuDNN
settings; exact cross-hardware reproducibility is not guaranteed.

In [ ]:
import importlib
runner_path = LOCAL_PROJECT / "repair_runner.py"
runner_path.write_text('"""Paired CNN78u optimizer diagnostic; training/validation only."""\nimport copy\nimport hashlib\nimport json\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Subset\n\nfrom us_pipeline.dataset import ImageDataset\nfrom us_pipeline.models import build_model\nfrom us_pipeline.reporting import binary_metrics, write_evaluation\nfrom us_pipeline.experiment import (make_loader, predict, save_json, save_checkpoint,\n                                    mirror, sync_tree, sync_cuda, runtime_info)\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef fixed_indices(n, fraction=.2, seed=69):\n    if not 0 < fraction <= 1 or n < 1:\n        raise ValueError(\'Invalid sampling contract\')\n    count = max(1, int(np.floor(n * fraction)))\n    return np.sort(np.random.default_rng(seed).choice(n, count, replace=False))\n\n\ndef probability_summary(p):\n    p = np.asarray(p, dtype=np.float64)\n    return dict(probability_std=float(p.std()), probability_min=float(p.min()),\n                probability_max=float(p.max()), unique_probabilities=len(np.unique(p)),\n                predicted_up_fraction=float((p >= .5).mean()))\n\n\ndef collapse_signal(health):\n    # Both constant predictions AND near-zero feature parameters are required.\n    # This is a numerical-failure guard, not a significance test for weak alpha.\n    return (health[\'train\'][\'probability_std\'] <= 1e-8\n            and health[\'validation\'][\'probability_std\'] <= 1e-8\n            and health[\'feature_parameter_max_abs\'] <= 1e-8)\n\n\ndef seed_all(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef probe_tensor(ds, indices):\n    rows = [ds[int(i)] for i in indices]\n    return torch.stack([r[0] for r in rows]), torch.tensor([r[1] for r in rows])\n\n\ndef health_check(model, probes, device, gradients, step):\n    was_training = model.training\n    model.eval(); result = {\'step\': step, \'gradient_norms_last_training_batch\': gradients}\n    with torch.no_grad():\n        for name, (x, y) in probes.items():\n            p = torch.cat([model(batch.to(device)).softmax(1)[:, 1].cpu()\n                           for batch in x.split(16)]).numpy()\n            result[name] = {**probability_summary(p),\n                            \'accuracy\': float(((p >= .5) == y.numpy()).mean())}\n        weights = {name: float(p.detach().abs().max()) for name,p in model.blocks.named_parameters()\n                   if name.endswith(\'weight\')}\n        result[\'feature_parameter_max_abs\'] = max(weights.values())\n        result[\'feature_weight_max_abs\'] = weights\n        result[\'batchnorm_scale_max_abs\'] = {str(i):float(m.weight.abs().max())\n            for i,m in enumerate(model.blocks) if isinstance(m, nn.BatchNorm2d)}\n        z = probes[\'train\'][0][:16].to(device); activation = {}\n        for i, layer in enumerate(model.blocks):\n            z = layer(z)\n            if isinstance(layer, nn.ReLU):\n                activation[str(i)] = {\'zero_fraction\':float((z==0).float().mean()),\n                    \'between_sample_std\':float(z.std(dim=0, unbiased=False).mean())}\n        result[\'relu_activations\'] = activation\n    model.train(was_training)\n    result[\'collapse_signal\'] = collapse_signal(result)\n    return result\n\n\ndef run_arm(prepared, root, persistent, indices, probe_ids, settings, weight_decay, smoke=False):\n    """Epoch-boundary resume; terminal collapsed/completed runs are read-only cached."""\n    root, persistent = Path(root), Path(persistent)\n    root.mkdir(parents=True, exist_ok=True); persistent.mkdir(parents=True, exist_ok=True)\n    train = ImageDataset(prepared, \'train\'); validation = ImageDataset(prepared, \'validation\')\n    if train.config[\'lookback\'] != 78 or train.config[\'horizon\'] != 1:\n        raise ValueError(\'Expected 78-bar input and next-bar target\')\n    config = train.config\n    contract = {\'settings\':settings,\'weight_decay\':weight_decay,\'config\':config,\'smoke\':smoke,\n                \'indices_sha256\':hashlib.sha256(indices.tobytes()).hexdigest(),\n                \'probes\':{k:v.tolist() for k,v in probe_ids.items()},\n                \'source_sha256\':sha(__file__),\n                \'data\':{n:sha(Path(prepared)/n) for n in [\'manifest.json\',\'features.npy\',\n                       \'multipliers.npy\',\'valid.npy\',\'train.parquet\',\'validation.parquet\']}}\n    signature = hashlib.sha256(json.dumps(contract,sort_keys=True).encode()).hexdigest()\n    if (root/\'contract.json\').exists():\n        if json.loads((root/\'contract.json\').read_text()) != contract:\n            raise ValueError(\'Run contract changed; use a new experiment ID\')\n    save_json(root/\'contract.json\', contract, persistent)\n    if (root/\'summary.json\').exists():\n        return json.loads((root/\'summary.json\').read_text())\n    seed_all(settings[\'seed\'])\n    model = build_model(config).to(device := torch.device(settings[\'device\']))\n    initial_sha = hashlib.sha256(b\'\'.join(p.detach().cpu().numpy().tobytes()\n                                         for p in model.parameters())).hexdigest()\n    optimizer = torch.optim.Adam(model.parameters(),lr=settings[\'learning_rate\'],weight_decay=weight_decay)\n    loader_settings = {**settings, \'amp\':False}\n    train_loader = make_loader(Subset(train, indices.tolist()), loader_settings, True)\n    val_view = Subset(validation, probe_ids[\'validation\'].tolist()) if smoke else validation\n    val_loader = make_loader(val_view, loader_settings, False)\n    probes = {\'train\':probe_tensor(train,probe_ids[\'train\']),\n              \'validation\':probe_tensor(validation,probe_ids[\'validation\'])}\n    prior = float(train.samples.iloc[indices].label.mean())\n    history=[]; health=[]; best=float(\'inf\'); first=1; steps=0; strikes=0\n    if (root/\'last.pt\').exists():\n        ck=torch.load(root/\'last.pt\',map_location=\'cpu\',weights_only=False)\n        if ck[\'signature\'] != signature: raise ValueError(\'Resume signature mismatch\')\n        model.load_state_dict(ck[\'model\']); optimizer.load_state_dict(ck[\'optimizer\'])\n        history=ck[\'history\']; health=ck[\'health\']; best=ck[\'best\']; steps=ck[\'steps\']\n        strikes=ck[\'strikes\']; first=ck[\'epoch\']+1\n        torch.set_rng_state(ck[\'torch_rng\'])\n        if device.type==\'cuda\': torch.cuda.set_rng_state_all(ck[\'cuda_rng\'])\n    else:\n        health.append(health_check(model,probes,device,{},0))\n    save_json(root/\'health.json\',health,persistent)\n    stopped=False; stop_reason=None\n    for epoch in range(first,settings[\'epochs\']+1):\n        model.train(); train_loader.generator.manual_seed(settings[\'seed\']+epoch)\n        if device.type==\'cuda\': torch.cuda.reset_peak_memory_stats(device)\n        sync_cuda(device); started=time.perf_counter(); total=correct=seen=0; monitor_seconds=0.\n        for batch_index,(x,y,_) in enumerate(train_loader):\n            x,y=x.to(device),y.to(device); optimizer.zero_grad(set_to_none=True)\n            logits=model(x); loss=nn.functional.cross_entropy(logits,y)\n            if not torch.isfinite(loss):\n                stopped=True; stop_reason=\'nonfinite_loss\'; break\n            loss.backward()\n            monitor=(steps+1)%settings[\'monitor_every\']==0 or batch_index+1==len(train_loader)\n            gradients={n:float(p.grad.detach().double().norm()) for n,p in model.named_parameters()\n                       if p.grad is not None} if monitor else {}\n            if monitor and not all(np.isfinite(list(gradients.values()))):\n                stopped=True; stop_reason=\'nonfinite_gradient\'; break\n            optimizer.step(); steps+=1\n            seen+=len(y); total+=float(loss.detach())*len(y); correct+=int((logits.argmax(1)==y).sum())\n            if monitor:\n                sync_cuda(device); t=time.perf_counter()\n                h=health_check(model,probes,device,gradients,steps); h[\'epoch\']=epoch\n                health.append(h); strikes=strikes+1 if h[\'collapse_signal\'] else 0\n                save_json(root/\'health.json\',health,persistent)\n                sync_cuda(device); monitor_seconds+=time.perf_counter()-t\n                print(json.dumps({\'epoch\':epoch,\'step\':steps,\'train_loss\':total/seen,\n                    \'probe_probability_std\':h[\'validation\'][\'probability_std\'],\n                    \'feature_max_abs\':h[\'feature_parameter_max_abs\'],\'collapse_strikes\':strikes}),flush=True)\n                if strikes>=3:\n                    stopped=True; stop_reason=\'three_consecutive_feature_collapse_checks\'; break\n        sync_cuda(device); train_wall=time.perf_counter()-started\n        frame,timing=predict(model,validation,val_loader,device,False,not smoke)\n        if not np.isfinite(frame.probability_up).all():\n            stopped=True; stop_reason=\'nonfinite_validation_probability\'\n            metrics={}\n        else:\n            metrics=binary_metrics(frame.label,frame.probability_up)\n        row={\'epoch\':epoch,\'steps_total\':steps,\'samples_seen\':seen,\'partial_epoch\':stopped,\n             \'train_loss\':total/seen if seen else None,\'train_accuracy\':correct/seen if seen else None,\n             \'train_wall_seconds_including_monitor\':train_wall,\'monitor_seconds\':monitor_seconds,\n             \'validation_seconds\':timing[\'end_to_end_seconds\'],\n             \'train_samples_per_second_including_monitor\':seen/train_wall,\n             \'gpu_peak_allocated_bytes\':torch.cuda.max_memory_allocated(device) if device.type==\'cuda\' else None,\n             **{\'val_\'+k:v for k,v in metrics.items() if not isinstance(v,list)}}\n        history.append(row)\n        ck={\'model\':model.state_dict(),\'optimizer\':optimizer.state_dict(),\'config\':config,\n            \'signature\':signature,\'history\':history,\'health\':health,\'steps\':steps,\'epoch\':epoch,\n            \'strikes\':strikes,\'torch_rng\':torch.get_rng_state(),\n            \'cuda_rng\':torch.cuda.get_rng_state_all() if device.type==\'cuda\' else None}\n        if not stopped and metrics[\'log_loss\']<best:\n            best=metrics[\'log_loss\']; ck[\'best\']=best\n            save_checkpoint(root/\'best.pt\',ck,persistent)\n            frame.to_parquet(root/\'best_validation_predictions.parquet\',index=False)\n            mirror(root/\'best_validation_predictions.parquet\',persistent)\n        ck[\'best\']=best\n        save_checkpoint(root/(\'stopped.pt\' if stopped else \'last.pt\'),ck,persistent)\n        save_json(root/\'history.json\',history,persistent)\n        save_json(root/\'health.json\',health,persistent)\n        if stopped: break\n    # A best pre-collapse checkpoint remains explicitly distinguished from terminal health.\n    evaluation=None\n    if (root/\'best_validation_predictions.parquet\').exists():\n        frame=pd.read_parquet(root/\'best_validation_predictions.parquet\')\n        evaluation=write_evaluation(frame,root/\'validation\',prior,\'validation\',{},\n            seed=settings[\'seed\'],bootstrap_repetitions=100 if smoke else 2000,smoke_only=smoke)\n        sync_tree(root/\'validation\',persistent/\'validation\')\n    result={\'status\':\'stopped_unhealthy\' if stopped else \'completed_budget\',\n            \'stop_reason\':stop_reason,\'smoke_only\':smoke,\'test_evaluated\':False,\n            \'weight_decay\':weight_decay,\'initial_parameters_sha256\':initial_sha,\n            \'train_subset_sha256\':contract[\'indices_sha256\'],\'training_samples\':len(indices),\n            \'training_prior\':prior,\'steps\':steps,\'history\':history,\n            \'final_health\':health[-1],\'runtime\':runtime_info(device),\n            \'best_validation_metrics\':evaluation[\'metrics\'] if evaluation else None,\n            \'interpretation\':\'Diagnostic only. Completing the budget is not proof of generalization.\'}\n    save_json(root/\'summary.json\',result,persistent)\n    return result\n')
importlib.invalidate_caches()
import repair_runner as repair

In [ ]:
import numpy as np, pandas as pd, torch
from IPython.display import display, Image
from datetime import datetime, timezone
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SMOKE_MODE = LOCAL_TEST  # Local QA only; never interpret smoke scores as performance.
if DEVICE != "cuda" and not SMOKE_MODE:
    raise RuntimeError("Select a GPU runtime before running the paired experiment")
torch.set_num_threads(min(4, os.cpu_count() or 1))
if DEVICE == "cuda":
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
print(repair.runtime_info(torch.device(DEVICE)))

## 5. Freeze the protocol and output folder
Normal runs retain `floor(0.2 * N)` eligible training windows uniformly without
replacement. The same indices are reused for all epochs and both arms; only their
order changes. No class balancing occurs. Validation is complete at each epoch.
A 20% subset with three epochs has about one fifth of the original update budget.

Set `RESUME=True` and the original `EXPERIMENT_ID` after a disconnect. All settings,
code, data, sampling and probe indices must match. A partly completed epoch is
repeated from the last completed epoch; a terminal collapsed run is not resumed.
Completed arms return cached summaries. Do not reuse this ID for a new experiment.
The two arms run sequentially to keep memory use manageable.

In [ ]:
SAMPLING_RATE = 0.2
SAMPLE_SEED = 69
SETTINGS = dict(epochs=1 if SMOKE_MODE else 3,
                batch_size=4 if SMOKE_MODE else 64,
                num_workers=0 if SMOKE_MODE else 2,
                learning_rate=0.001, seed=69,
                monitor_every=1 if SMOKE_MODE else 500, device=DEVICE)
RESUME = False
EXPERIMENT_ID = ("repair_local_qa" if LOCAL_TEST else
                "cnn78u_repair_20pct_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_utc"))
PERSISTENT = DRIVE_DEMO / "repair_runs" / EXPERIMENT_ID
LOCAL_REPAIR = LOCAL_CACHE / "repair"
if RESUME:
    if not (PERSISTENT / "sampling.json").exists():
        raise FileNotFoundError("No sampling contract for this experiment ID")
    shutil.copytree(PERSISTENT, LOCAL_REPAIR)
else:
    if PERSISTENT.exists():
        raise FileExistsError("Choose a new experiment ID or explicitly resume")
    PERSISTENT.mkdir(parents=True)
    LOCAL_REPAIR.mkdir()
print("Persistent output:", PERSISTENT)
display(pd.Series(SETTINGS))

## 6. Persist the fixed training subset and fixed diagnostic probes
The probes are 128 seeded examples from the selected training subset and 128
seeded validation examples. They detect numerical failure, not statistical alpha.
Full validation remains the source of epoch metrics. Hashes and original row indices
are saved so later models can use exactly the same subset. For the current bundle,
the normal training subset contains 344,165 of 1,720,827 eligible examples.

In [ ]:
train = repair.ImageDataset(PREPARED, "train")
validation = repair.ImageDataset(PREPARED, "validation")
indices = repair.fixed_indices(len(train), SAMPLING_RATE, SAMPLE_SEED)
if SMOKE_MODE:
    indices = indices[:8]
probe_rng = np.random.default_rng(SAMPLE_SEED + 1)
probe_ids = {"train":np.sort(probe_rng.choice(indices, min(128, len(indices)), replace=False)),
             "validation":np.sort(probe_rng.choice(len(validation), 8 if SMOKE_MODE else min(128,len(validation)), replace=False))}
contract = dict(requested_sampling_rate=SAMPLING_RATE, actual_fraction=len(indices)/len(train),
                sampling_seed=SAMPLE_SEED, samples=len(indices), eligible_samples=len(train),
                subset_sha256=hashlib.sha256(indices.tobytes()).hexdigest(),
                train_index_file_sha256=repair.sha(PREPARED / "train.parquet"),
                smoke_only=SMOKE_MODE)
if RESUME:
    assert json.loads((LOCAL_REPAIR / "sampling.json").read_text()) == contract
    assert np.array_equal(np.load(LOCAL_REPAIR / "train_indices.npy"), indices)
else:
    np.save(LOCAL_REPAIR / "train_indices.npy", indices)
    repair.mirror(LOCAL_REPAIR / "train_indices.npy", PERSISTENT)
    repair.save_json(LOCAL_REPAIR / "sampling.json", contract, PERSISTENT)
    np.savez(LOCAL_REPAIR / "probe_indices.npz", **probe_ids)
    repair.mirror(LOCAL_REPAIR / "probe_indices.npz", PERSISTENT)
display(pd.Series(contract))
print("Retained training up fraction:", train.samples.iloc[indices].label.mean())
print("Validation samples:", len(validation) if not SMOKE_MODE else len(probe_ids["validation"]))

## 7. Original optimizer control: weight decay = 0.0001
Every 500 updates and at each epoch end, record probability dispersion, convolution
and BatchNorm weight magnitudes, ReLU activity, and gradient norms from the latest
training batch. The probes run in eval mode and do not update BatchNorm statistics.

Stop after three consecutive checks where **both** probe probability standard
deviations are at most 1e-8 **and** all convolution weights / BatchNorm scales have
maximum absolute magnitude at most 1e-8. Biases are excluded from that magnitude
check because a collapsed network can retain constant offsets. Nonfinite loss or
monitored gradients also stop the arm. These conservative guards do not detect
all possible failures; low discrimination still requires review.

A stopped arm saves `stopped.pt`. Each completed epoch saves `last.pt`; the best
completed epoch by full-validation log loss saves `best.pt`. The comparison flags
terminal health separately from the best pre-collapse checkpoint.

In [ ]:
control = repair.run_arm(PREPARED, LOCAL_REPAIR / "wd_1e-4", PERSISTENT / "wd_1e-4",
                         indices, probe_ids, SETTINGS, weight_decay=0.0001, smoke=SMOKE_MODE)
display(pd.DataFrame(control["history"]))
print("Control status:", control["status"], control["stop_reason"])

## 8. Repair candidate: weight decay = 0
Only weight decay changes. Dropout remains 0.5, unlike the earlier small-sample
memorization check. This arm starts from the same fresh initialization as the
control; it does not fine-tune the collapsed checkpoint.

In [ ]:
candidate = repair.run_arm(PREPARED, LOCAL_REPAIR / "wd_0", PERSISTENT / "wd_0",
                           indices, probe_ids, SETTINGS, weight_decay=0.0, smoke=SMOKE_MODE)
assert control["initial_parameters_sha256"] == candidate["initial_parameters_sha256"]
assert control["train_subset_sha256"] == candidate["train_subset_sha256"]
display(pd.DataFrame(candidate["history"]))
print("Candidate status:", candidate["status"], candidate["stop_reason"])

## 9. Compare validation and training health
Compare validation log loss, balanced accuracy and AUC against the training-prior
baseline. Accuracy above 50% alone is insufficient. Completing the budget only
means the stop guard did not trigger; it does not certify useful predictive signal.
If one arm stops early, report its smaller update budget explicitly.

The validation report uses the best completed epoch, while terminal health uses
the final monitored model. If no healthy epoch completed, no best report exists.
Baseline tables and reliability/confusion/ROC plots are saved under each arm's
`validation/` directory. No result here is a new held-out test result.

In [ ]:
rows=[]
for name, result in [("Original WD 1e-4", control),("Candidate WD 0", candidate)]:
    metrics=result["best_validation_metrics"] or {}
    rows.append(dict(arm=name, status=result["status"], steps=result["steps"],
        validation_accuracy=metrics.get("accuracy"), balanced_accuracy=metrics.get("balanced_accuracy"),
        roc_auc=metrics.get("roc_auc"), log_loss=metrics.get("log_loss"),
        terminal_probe_std=result["final_health"]["validation"]["probability_std"],
        terminal_feature_max_abs=result["final_health"]["feature_parameter_max_abs"],
        training_wall_seconds=sum(r["train_wall_seconds_including_monitor"] for r in result["history"]),
        validation_seconds=sum(r["validation_seconds"] for r in result["history"])))
comparison=pd.DataFrame(rows)
display(comparison)
comparison.to_csv(LOCAL_REPAIR / "comparison.csv", index=False)
repair.mirror(LOCAL_REPAIR / "comparison.csv", PERSISTENT)
for arm in ["wd_1e-4","wd_0"]:
    report = LOCAL_REPAIR / arm / "validation"
    if report.exists():
        print(arm)
        display(pd.read_csv(report / "baselines.csv"))
        display(Image(filename=str(report / "evaluation.png")))

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for arm in ["wd_1e-4", "wd_0"]:
    health = json.loads((LOCAL_REPAIR / arm / "health.json").read_text())
    steps = [r["step"] for r in health]
    axes[0].plot(steps, [max(r["feature_parameter_max_abs"],1e-45) for r in health], label=arm)
    axes[1].plot(steps, [max(r["validation"]["probability_std"],1e-12) for r in health], label=arm)
    history = json.loads((LOCAL_REPAIR / arm / "history.json").read_text())
    axes[2].plot([r["steps_total"] for r in history], [r.get("val_log_loss",np.nan) for r in history], marker="o", label=arm)
for ax,title in zip(axes,["Feature weights: max absolute value", "Validation probe probability std", "Full-validation log loss"]):
    ax.set_title(title); ax.set_xlabel("Optimizer updates"); ax.legend()
axes[0].set_yscale("log"); axes[1].set_yscale("log")
if SMOKE_MODE:
    axes[2].set_title("Smoke validation log loss")
    fig.suptitle("SMOKE CHECK ONLY: not a performance estimate")
fig.tight_layout()
fig.savefig(LOCAL_REPAIR / "health_comparison.png",dpi=140)
plt.close(fig)
repair.mirror(LOCAL_REPAIR / "health_comparison.png",PERSISTENT)
display(Image(filename=str(LOCAL_REPAIR / "health_comparison.png")))

## 10. Save and decide the next step
- If the control collapses and the candidate remains input-sensitive, the paired
  experiment supports removing this decay configuration as a repair under this
  protocol. It does not isolate effects across all datasets, budgets or seeds.
- If both remain healthy but validation is near chance, investigate signal and
  representation before claiming success.
- If both collapse, investigate learning rate, activations and optimization using
  further controlled experiments.
- If neither collapses within three epochs, the smaller update budget may not
  reproduce the original late failure. Do not claim weight decay was cleared or
  proved causal. Any extended budget should be predeclared for both arms.

Stop here. Residual connections, depth changes and other architectures belong to
a later experiment after establishing a functioning baseline. Preserve this fixed
subset and the original failed run. Use a new untouched period for final evaluation.

In [ ]:
summary = dict(experiment_id=EXPERIMENT_ID, smoke_only=SMOKE_MODE, test_evaluated=False,
               same_initialization=True, same_training_subset=True,
               arms={"wd_1e-4":control["status"],"wd_0":candidate["status"]})
repair.save_json(LOCAL_REPAIR / "experiment_summary.json",summary,PERSISTENT)
archive=Path(shutil.make_archive(str(LOCAL_CACHE / "repair_results"),"zip",root_dir=LOCAL_REPAIR))
shutil.copy2(archive,PERSISTENT / "repair_results.zip")
print("Saved diagnostic results:",PERSISTENT)
display(pd.Series(summary))